# Getting Started with SynthBench

This notebook walks you through running your first synthetic survey benchmark,
inspecting the results, and comparing models — all hands-on.

**Target audience**: Data scientists who are skeptical of synthetic surveys
and want to verify the methodology before trusting it.

**What you'll learn**:
1. Run a benchmark with zero API keys (random provider)
2. Read and interpret the JSON results
3. Display the score card
4. Compare two result files with statistical significance testing
5. Run with a real API key (OpenRouter)
6. Explore topic-specific subsets (political vs consumer)
7. Plug in your own REST API via HttpProvider

## 1. Install SynthBench

Install from the repo. This gives you the `synthbench` CLI and Python library.

In [ ]:
!pip install -e ".[dev]" --quiet

## 2. Run Your First Benchmark (No API Key Needed)

The `random` provider generates uniform random answers — no API key required.
This is the **floor baseline**: any real model should beat this.

We use `--suite smoke` (28 questions) for speed.

In [ ]:
!synthbench run --provider random --suite smoke --samples 10 --output ./my_results

## 3. Inspect the JSON Results

Every run produces a JSON file with the full results. Let's load it and
look at the structure.

In [ ]:
import json
from pathlib import Path

# Find the most recent result file
result_files = sorted(Path("./my_results").glob("*.json"))
assert result_files, "No result files found — did the benchmark run?"

result_file = result_files[-1]
print(f"Loading: {result_file.name}")

with open(result_file) as f:
    data = json.load(f)

# Top-level structure
print(f"\nBenchmark: {data['benchmark']} v{data['version']}")
print(f"Provider:  {data['config']['provider']}")
print(f"Dataset:   {data['config']['dataset']}")
print(f"Questions: {data['aggregate']['n_questions']}")

# Scores
print(f"\n--- Scores ---")
for metric, value in data["scores"].items():
    print(f"  {metric}: {value:.4f}")

# Look at one question in detail
q = data["per_question"][0]
print(f"\n--- Example Question ---")
print(f"  Q: {q['text']}")
print(f"  Human dist:  {q['human_distribution']}")
print(f"  Model dist:  {q['model_distribution']}")
print(f"  JSD: {q['jsd']:.4f}  |  tau: {q['kendall_tau']:.4f}")

## 4. Display the Score Card

The `synthbench report` command regenerates a readable score card from any
JSON result file.

In [ ]:
import subprocess

result = subprocess.run(
    ["synthbench", "report", str(result_file)], capture_output=True, text=True
)
print(result.stdout)

### Understanding the Scores

All scores range from **0** (no resemblance to humans) to **1** (indistinguishable).

| Score | What it measures |
|-------|------------------|
| **P_dist** | How closely does the AI's answer distribution match real humans? |
| **P_rank** | Does the AI get the preference ordering right? |
| **P_refuse** | Does the AI refuse to answer at appropriate rates? |
| **P_cond** | Does persona conditioning (e.g., "respond as a 65-year-old") actually work? |
| **P_sub** | Is the AI equally accurate across all demographics? |
| **SPS** | Overall score — average of all available metrics. |

## 5. Compare Two Results

Let's run a second benchmark with the `majority` baseline (always picks the
most popular answer) and compare it to our random run.

The comparison includes a paired bootstrap significance test — so you'll
know if the difference is real or just noise.

In [ ]:
!synthbench run --provider majority --suite smoke --samples 10 --output ./my_results

In [ ]:
# Find the two result JSON files (random and majority)
all_jsons = sorted(Path("./my_results").glob("*.json"))
print(f"Result files: {[f.name for f in all_jsons]}")

# Compare the first two
if len(all_jsons) >= 2:
    !synthbench compare {str(all_jsons[0])} {str(all_jsons[1])}
else:
    print("Need at least 2 result files to compare.")

## 6. Run with a Real API Key (OpenRouter)

Now let's benchmark an actual LLM. [OpenRouter](https://openrouter.ai/) is
an easy way to access many models with one API key.

Set your `OPENROUTER_API_KEY` environment variable before running.
You can also use `--provider raw-anthropic` (needs `ANTHROPIC_API_KEY`) or
`--provider raw-openai` (needs `OPENAI_API_KEY`).

In [ ]:
import os

# Uncomment and set your key:
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."

if os.environ.get("OPENROUTER_API_KEY"):
    !synthbench run \
        --provider openrouter \
        --model anthropic/claude-haiku-4-5-20251001 \
        --suite smoke \
        --samples 10 \
        --output ./my_results
else:
    print("Skipped — set OPENROUTER_API_KEY to run with a real model.")
    print("Get a key at https://openrouter.ai/keys")

## 7. Topic Analysis — Consumer vs Political

SynthBench includes topic-tagged question subsets. Models often perform
differently on politically sensitive vs consumer-oriented questions.

Let's run the random baseline on each topic to see the structure.

In [ ]:
# Run consumer subset
!synthbench run --provider random --topic consumer --samples 10 --output ./my_results

# Run political subset
!synthbench run --provider random --topic political --samples 10 --output ./my_results

In [ ]:
# Compare topic scores
topic_results = sorted(Path("./my_results").glob("*.json"))

for f in topic_results:
    with open(f) as fh:
        d = json.load(fh)
    topic = d.get("config", {}).get("topic", "overall")
    sps = d.get("scores", {}).get(
        "sps", d.get("aggregate", {}).get("composite_parity", 0)
    )
    provider = d["config"]["provider"]
    print(f"{provider:20s} topic={topic:10s}  SPS={sps:.4f}")

## 8. Plug In Your Own Provider (HttpProvider)

Have your own survey respondent API? SynthBench can benchmark any REST
endpoint that accepts this POST format:

```json
{
  "question": "Do you support policy X?",
  "options": ["Strongly favor", "Favor", "Oppose", "Strongly oppose"],
  "persona": {"demographics": {"age": "18-29"}, ...}
}
```

Your endpoint returns either:
- **Single response**: `{"selected_option": "Favor"}`
- **Distribution**: `{"probabilities": [0.1, 0.4, 0.3, 0.2]}`

In [ ]:
# Example: benchmark your own endpoint
# Replace the URL with your actual API endpoint.

# !synthbench run \
#     --provider http \
#     --url https://your-api.example.com/survey \
#     --suite smoke \
#     --samples 30 \
#     --output ./my_results

print("Uncomment the command above and replace the URL with your endpoint.")
print("\nYour endpoint should accept POST requests with JSON body:")
print('  {"question": "...", "options": [...], "persona": {...}}')
print("\nAnd return one of:")
print('  {"selected_option": "..."}')
print('  {"probabilities": [0.1, 0.4, 0.3, 0.2]}')

## Next Steps

- **Full benchmark**: Replace `--suite smoke` with `--suite core` (200 questions)
  or remove `--suite` entirely for all 1,498 questions.
- **More samples**: Increase `--samples 30` (or higher) for more stable distributions.
- **Demographic evaluation**: Add `--demographics AGE,POLIDEOLOGY` to see
  P_cond and P_sub scores.
- **Replication**: Use `synthbench replicate --n-runs 5` to check score stability.
- **Leaderboard**: Use `synthbench publish` to generate an HTML leaderboard page.

See [METHODOLOGY.md](../METHODOLOGY.md) for the full scoring framework.